# nn.Linear et nn.Module : calculer les deux scores

Remplacer les scores choisis directement par une règle dont nous pouvons lire les poids.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#construire-un-module).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Reprenons les trois lignes RMS/crête. Nous voulons maintenant calculer les scores, plutôt que les écrire directement. Les poids seront d’abord fixés à la main.

| Sortie | Poids RMS | Poids facteur de crête | Biais |
| --- | --- | --- | --- |
| Voix, colonne 0 | 0 | 1 | 0 |
| Percussion, colonne 1 | 0 | −1 | 0 |

Le premier extrait donne 0×0,1 + 1×1,5 + 0 = 1,5 pour la voix, et −1,5 pour la percussion.

nn.Linear(2,2) signifie deux mesures d’entrée et deux scores de sortie. Son tableau weight contient une ligne de poids par sortie, donc une forme (2,2). Le biais possède deux valeurs.

from torch import nn importe le paquet des couches sous le nom nn. Un module hérite de nn.Module, la classe PyTorch prévue pour ces modèles.

__init__ prépare l’objet une seule fois ; self désigne cet objet. super().__init__ initialise la partie PyTorch qui enregistre les paramètres.

L’attribut self.linear conserve la couche dans le module. forward reçoit ensuite les entrées de chaque appel. modele(X) utilise cette méthode avec les mécanismes de PyTorch.

Linear choisit normalement ses premiers poids aléatoirement. Nous les remplaçons pour reproduire le tableau. no_grad suspend ici le suivi des opérations pendant cette installation, pas pendant les futurs calculs.

## Démonstration 1 · Définir la règle, puis inspecter un score

Suivre le passage de deux mesures à deux sorties.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn



### Définir la règle avant de l’utiliser

class Classifieur(nn.Module) crée notre classe de modèle. La définition prépare des méthodes ; elle ne calcule pas encore les scores.

Chaque instance reçoit sa propre couche. L’affectation à self.linear permet à PyTorch de retrouver ses poids et son biais.

self.linear devient un attribut du modèle. Linear(2,2) transforme deux entrées en deux sorties ; il enregistre ses propres poids et son biais.

forward ne remplace pas la cible et ne modifie pas les poids. Il transmet les trois lignes à la même couche.

In [4]:
class Classifieur(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 2)

    def forward(self, entrees):
        return self.linear(entrees)



### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
X = torch.tensor([[0.1, 1.5], [0.2, 4.5], [0.15, 2.5]], dtype=torch.float32)
cibles = torch.tensor([0, 1, 0], dtype=torch.long)


In [6]:
observer('X', X)
observer('cibles', cibles)

X forme (3, 2) type torch.float32 requires_grad False
  valeurs : [[0.10000000149011612, 1.5], [0.20000000298023224, 4.5], [0.15000000596046448, 2.5]]
cibles forme (3,) type torch.int64 requires_grad False
  valeurs : [0, 1, 0]


### Calculer l’état suivant

Classifieur() appelle __init__. copy_ recopie les nombres dans les poids existants ; zero_ met les deux biais à zéro. Le suffixe _ indique ici une modification en place.

In [7]:
modele = Classifieur()


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
with torch.no_grad():
    modele.linear.weight.copy_(torch.tensor([[0., 1.], [0., -1.]]))
    modele.linear.bias.zero_()


### Calculer l’état suivant

modele(X) applique forward à ce tableau. Les paramètres du modèle sont lus, et ne sont pas encore ajustés.

In [9]:
scores = modele(X)


In [10]:
observer('scores', scores)

scores forme (3, 2) type torch.float32 requires_grad True
  valeurs : [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]]


### Calculer l’état suivant

X[0] contient [0,1;1,5]. weight[0] contient [0;1]. Le produit élémentaire donne [0;1,5], puis sum additionne ces contributions.

In [11]:
produits = X[0] * modele.linear.weight[0]
score_manuel = produits.sum() + modele.linear.bias[0]


In [12]:
observer('produits', produits)
observer('score_manuel', score_manuel)

produits forme (2,) type torch.float32 requires_grad True
  valeurs : [0.0, 1.5]
score_manuel forme () type torch.float32 requires_grad True
  valeurs : 1.5


### Lire les sorties

named_parameters fournit les noms et objets de paramètres. La compréhension sert uniquement à afficher leurs formes.

In [13]:
print("poids", modele.linear.weight.tolist(), "forme", tuple(modele.linear.weight.shape))
print("produits voix A", produits.tolist(), "somme", score_manuel.item())
print("scores", scores.tolist(), "forme", tuple(scores.shape))
print("parametres", [(nom, tuple(p.shape)) for nom, p in modele.named_parameters()])

poids [[0.0, 1.0], [0.0, -1.0]] forme (2, 2)
produits voix A [0.0, 1.5] somme 1.5
scores [[1.5, -1.5], [4.5, -4.5], [2.5, -2.5]] forme (3, 2)
parametres [('linear.weight', (2, 2)), ('linear.bias', (2,))]


Les mêmes poids retrouvent les scores déjà étudiés. L’erreur sur la percussion vient désormais d’une règle identifiable, que nous pourrons ajuster.

Une seule couche linéaire ne représente pas toutes les frontières possibles. Une activation ReLU conserve les nombres positifs et remplace les négatifs par zéro.

## Démonstration 2 · Le programme du chapitre

Expérience indépendante : une activation sur trois nombres. La sortie est [0,0,3] ; le tenseur x reste inchangé.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
import torch


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [15]:
x = torch.tensor([-2., 0., 3.])
y = torch.relu(x)


In [16]:
observer('x', x)
observer('y', y)

x forme (3,) type torch.float32 requires_grad False
  valeurs : [-2.0, 0.0, 3.0]
y forme (3,) type torch.float32 requires_grad False
  valeurs : [0.0, 0.0, 3.0]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [17]:
print("entree", x.tolist(), "sortie", y.tolist())

entree [-2.0, 0.0, 3.0] sortie [0.0, 0.0, 3.0]


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Calculez le premier score d’une couche linéaire avec ces poids et ce biais.

In [18]:
exo_x = torch.tensor([.1,1.5])
exo_w = torch.tensor([0.,1.])
exo_b = torch.tensor(0.)
exo_score = ...  # Remplacer les points par votre réponse.
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score) - float((exo_x*exo_w).sum()+exo_b)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Faites les deux produits, puis leur somme et enfin le biais.

</details>

### Exercice 2

Nous changeons seulement le poids du premier descripteur, de 0 à 2. Quel score obtient-on ?

In [19]:
exo_x = torch.tensor([.1,1.5])
exo_w = torch.tensor([2.,1.])
exo_score = ...  # Remplacer les points par votre réponse.
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score)-float((exo_x*exo_w).sum())) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le nouveau terme vaut 0,1×2. La seconde contribution reste 1,5.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.tensor([.1,1.5])
exo_w = torch.tensor([0.,1.])
exo_b = torch.tensor(0.)
exo_score = 1.5
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score) - float((exo_x*exo_w).sum()+exo_b)) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.tensor([.1,1.5])
exo_w = torch.tensor([2.,1.])
exo_score = 1.7
if exo_score is Ellipsis or exo_score is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_score)-float((exo_x*exo_w).sum())) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [20]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()